# 05 — Random vs Sieve

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** compare deterministic sieve filtering against random removal with matched retention.

Notebook 01 measured residue constraints.  
Notebook 02 measured gap structure.  
Notebook 03 measured density scale.  
Notebook 04 measured sieve mechanism.  
Notebook 05 adds a random-control test.

Core claim:

> Same count does not imply same structure.

Random removal can match density, but it does not recover prime structure.

## 0. Setup

Artifact structure:

```text
05_random_vs_sieve/
├── data/
├── docs/
├── figures/
└── tex/
```

Root export:

```text
05_random_vs_sieve_export.zip
```

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "05_random_vs_sieve"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]
NOTEBOOK_TITLE = "Random vs Sieve"

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
FIG_DIR = OUT / "figures"
TEX_DIR = OUT / "tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact directory: {OUT.resolve()}")

## 1. Premise

This notebook compares three sets:

1. **Sieve / prime set**  
   Deterministically retained by divisibility constraints.

2. **Random-count-matched set**  
   Same final count as the prime set, sampled randomly from \(2,\dots,N\).

3. **Random-layer-matched set**  
   At each sieve layer, remove the same number of candidates as the real sieve, but choose candidates randomly.

The strongest comparison is:

> matched retention does not imply matched structure.

## 2. Core definitions

Let:

\[
P_N = \{p : p \le N\}
\]

For a candidate set \(S\), define recovery:

\[
CGCS_{\mathrm{recovery}}(S) =
\frac{|S \cap P_N|}{|P_N|}
\]

and drift:

\[
drift_{\mathrm{recovery}}(S) =
1 - CGCS_{\mathrm{recovery}}(S)
\]

A random set may match \(|P_N|\), but it should not match the structure of \(P_N\).

In [ ]:
# Parameters

N_MAX = 200_000
RANDOM_SEED = 9423

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
    "comparison": "sieve primes vs random-count-matched and random-layer-matched controls",
}

params

## 3. Generate reference primes and sieve-layer removals

Use the same layer-by-layer sieve logic from Notebook 04, while recording how many candidates each prime filter removes.

In [ ]:
def simple_sieve(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=int)
    s = np.ones(n + 1, dtype=bool)
    s[:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if s[i]:
            s[i*i:n+1:i] = False
    return np.nonzero(s)[0]

reference_primes = simple_sieve(N_MAX)
filter_primes = reference_primes[reference_primes <= int(math.sqrt(N_MAX))]
universe = np.arange(2, N_MAX + 1)

# Real layer-by-layer sieve.
candidate_mask = np.zeros(N_MAX + 1, dtype=bool)
candidate_mask[2:] = True
all_numbers = np.arange(N_MAX + 1)

sieve_rows = []
for layer_index, q in enumerate(filter_primes, start=1):
    before_count = int(candidate_mask.sum())

    remove_mask = candidate_mask.copy()
    remove_mask[:q+1] = False
    remove_mask &= (all_numbers % q == 0)

    removed_count = int(remove_mask.sum())
    candidate_mask[remove_mask] = False
    after_count = int(candidate_mask.sum())

    sieve_rows.append({
        "layer": layer_index,
        "filter_prime_q": int(q),
        "candidate_count_before": before_count,
        "removed_count": removed_count,
        "candidate_count_after": after_count,
        "retention_share": after_count / len(universe),
    })

sieve_layers_df = pd.DataFrame(sieve_rows)

sieve_set = np.nonzero(candidate_mask)[0]
sieve_set = sieve_set[sieve_set >= 2]
exact_match = np.array_equal(sieve_set, reference_primes)

summary = {
    "n_max": int(N_MAX),
    "universe_count": int(len(universe)),
    "prime_count": int(len(reference_primes)),
    "filter_prime_count": int(len(filter_primes)),
    "exact_sieve_match": bool(exact_match),
    "first_primes": reference_primes[:10].tolist(),
    "last_primes": reference_primes[-10:].tolist(),
}

summary

## 4. Construct random controls

### Random-count-matched

Choose \(|P_N|\) random integers from \(2,\dots,N\).

### Random-layer-matched

Apply the same number of removals at each sieve layer, but remove random retained candidates instead of multiples.

In [ ]:
# Random-count-matched set.
random_count_set = np.sort(rng.choice(universe, size=len(reference_primes), replace=False))

# Random-layer-matched set.
random_layer_mask = np.zeros(N_MAX + 1, dtype=bool)
random_layer_mask[2:] = True

random_layer_rows = []

for _, row in sieve_layers_df.iterrows():
    layer = int(row["layer"])
    q = int(row["filter_prime_q"])
    before_count = int(random_layer_mask.sum())
    target_remove = int(row["removed_count"])

    retained = np.nonzero(random_layer_mask)[0]
    retained = retained[retained >= 2]

    remove_count = min(target_remove, len(retained))
    if remove_count > 0:
        remove_values = rng.choice(retained, size=remove_count, replace=False)
        random_layer_mask[remove_values] = False

    after_count = int(random_layer_mask.sum())

    random_layer_rows.append({
        "layer": layer,
        "filter_prime_q": q,
        "candidate_count_before": before_count,
        "target_removed_count": target_remove,
        "actual_removed_count": before_count - after_count,
        "candidate_count_after": after_count,
        "retention_share": after_count / len(universe),
    })

random_layer_df = pd.DataFrame(random_layer_rows)

random_layer_set = np.nonzero(random_layer_mask)[0]
random_layer_set = random_layer_set[random_layer_set >= 2]

sets = {
    "sieve_primes": reference_primes,
    "random_count_matched": random_count_set,
    "random_layer_matched": random_layer_set,
}

{key: len(value) for key, value in sets.items()}

## 5. Measurement functions

Compare:

- recovery against the prime set
- residue distribution modulo 6
- gap statistics
- density by scale

In [ ]:
prime_set = set(reference_primes.tolist())

def recovery_metrics(values: np.ndarray, label: str) -> dict:
    value_set = set(values.tolist())
    tp = len(value_set & prime_set)
    fp = len(value_set - prime_set)
    fn = len(prime_set - value_set)
    recovery = tp / len(prime_set) if prime_set else float("nan")
    precision = tp / len(value_set) if value_set else float("nan")
    drift = 1.0 - recovery
    return {
        "set": label,
        "count": int(len(values)),
        "true_prime_overlap": int(tp),
        "false_positive_count": int(fp),
        "false_negative_count": int(fn),
        "cgcs_recovery": float(recovery),
        "precision_against_primes": float(precision),
        "drift_recovery": float(drift),
    }

def residue_df(values: np.ndarray, label: str) -> pd.DataFrame:
    counts = np.bincount(values % 6, minlength=6)
    total = counts.sum()
    return pd.DataFrame({
        "set": label,
        "residue_mod_6": np.arange(6),
        "count": counts,
        "share": counts / total if total else np.zeros(6),
    })

def gap_summary(values: np.ndarray, label: str) -> dict:
    sorted_values = np.sort(values)
    gaps = np.diff(sorted_values)
    if len(gaps) == 0:
        return {
            "set": label,
            "gap_count": 0,
            "mean_gap": float("nan"),
            "median_gap": float("nan"),
            "max_gap": float("nan"),
            "even_gap_share": float("nan"),
        }
    return {
        "set": label,
        "gap_count": int(len(gaps)),
        "mean_gap": float(np.mean(gaps)),
        "median_gap": float(np.median(gaps)),
        "max_gap": int(np.max(gaps)),
        "even_gap_share": float(np.mean(gaps % 2 == 0)),
    }

def density_by_scale(values: np.ndarray, label: str, scales: np.ndarray) -> pd.DataFrame:
    values = np.sort(values)
    rows = []
    for x in scales:
        rows.append({
            "set": label,
            "x": int(x),
            "count_leq_x": int(np.searchsorted(values, x, side="right")),
            "density_leq_x": float(np.searchsorted(values, x, side="right") / max(1, x - 1)),
        })
    return pd.DataFrame(rows)

set_metrics_df = pd.DataFrame([
    recovery_metrics(values, label)
    for label, values in sets.items()
])

residue_comparison_df = pd.concat([
    residue_df(values, label)
    for label, values in sets.items()
], ignore_index=True)

gap_summary_df = pd.DataFrame([
    gap_summary(values, label)
    for label, values in sets.items()
])

scales = np.unique(np.logspace(2, np.log10(N_MAX), 70).astype(int))
density_df = pd.concat([
    density_by_scale(values, label, scales)
    for label, values in sets.items()
], ignore_index=True)

set_metrics_df, gap_summary_df.head()

## 6. CGCS summary

The main score is recovery against the true prime set:

\[
CGCS_{\mathrm{recovery}}(S) =
\frac{|S\cap P_N|}{|P_N|}
\]

Expected:

- sieve: \(1.0\)
- random-count-matched: low
- random-layer-matched: low

In [ ]:
sieve_recovery = float(set_metrics_df.loc[set_metrics_df["set"] == "sieve_primes", "cgcs_recovery"].iloc[0])
random_count_recovery = float(set_metrics_df.loc[set_metrics_df["set"] == "random_count_matched", "cgcs_recovery"].iloc[0])
random_layer_recovery = float(set_metrics_df.loc[set_metrics_df["set"] == "random_layer_matched", "cgcs_recovery"].iloc[0])

random_count_drift = 1.0 - random_count_recovery
random_layer_drift = 1.0 - random_layer_recovery

cgcs = {
    "score": sieve_recovery,
    "definition": "CGCS_recovery(S) = |S ∩ P_N| / |P_N|",
    "interpretation": "Sieve should score 1.0; random controls should score lower despite matched counts.",
}

measurement = {
    "sieve_recovery": sieve_recovery,
    "random_count_recovery": random_count_recovery,
    "random_layer_recovery": random_layer_recovery,
    "random_count_drift": random_count_drift,
    "random_layer_drift": random_layer_drift,
    "same_count_random_count_matched": int(len(random_count_set) == len(reference_primes)),
    "same_final_count_random_layer_matched": int(len(random_layer_set) == len(reference_primes)),
}

measurement

## 7. Figure 1 — residue comparison modulo 6

The sieve/prime set is constrained to \(1,5 \pmod 6\) except for \(2,3\).  
Random controls spread across residue classes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for label in ["sieve_primes", "random_count_matched", "random_layer_matched"]:
    sub = residue_comparison_df[residue_comparison_df["set"] == label]
    ax.plot(sub["residue_mod_6"], sub["share"], marker="o", label=label)

ax.set_title("Residue distribution modulo 6")
ax.set_xlabel("residue mod 6")
ax.set_ylabel("share")
ax.set_xticks(np.arange(6))
ax.legend()
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_residue_comparison_mod6.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

## 8. Figure 2 — gap histogram comparison

Prime gaps are structured.  
Random gaps reflect matched count, not matched prime structure.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for label, values in sets.items():
    gaps = np.diff(np.sort(values))
    gaps = gaps[gaps <= 100]  # focus visible range
    ax.hist(gaps, bins=50, alpha=0.45, label=label)

ax.set_title("Gap histogram comparison (gaps ≤ 100)")
ax.set_xlabel("gap")
ax.set_ylabel("frequency")
ax.legend()
ax.grid(True, alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_gap_histogram_comparison.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

## 9. Figure 3 — density by scale

Random-count-matched can match final count, but density by scale can drift.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for label in ["sieve_primes", "random_count_matched", "random_layer_matched"]:
    sub = density_df[density_df["set"] == label]
    ax.plot(sub["x"], sub["count_leq_x"], label=label)

ax.set_xscale("log")
ax.set_title("Count by scale")
ax.set_xlabel("x")
ax.set_ylabel("count ≤ x")
ax.legend()
ax.grid(True, alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_density_by_scale.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

## 10. Figure 4 — layer retention comparison

The random-layer-matched control follows the same removal count per layer.  
It matches retention by design, but not structure.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    sieve_layers_df["layer"],
    sieve_layers_df["retention_share"],
    marker="o",
    label="sieve retention",
)
ax.plot(
    random_layer_df["layer"],
    random_layer_df["retention_share"],
    marker="o",
    label="random layer-matched retention",
)

ax.set_title("Layer retention comparison")
ax.set_xlabel("layer")
ax.set_ylabel("retention share")
ax.legend()
ax.grid(True, alpha=0.3)

fig4_path = FIG_DIR / f"{NOTEBOOK_NUM}_layer_retention_comparison.png"
fig.savefig(fig4_path, dpi=180, bbox_inches="tight")
plt.show()

fig4_path

## 11. Figure 5 — recovery score comparison

Sieve recovers the prime set.  
Random controls do not.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(set_metrics_df["set"], set_metrics_df["cgcs_recovery"])
ax.set_ylim(0, 1.05)
ax.set_title("Prime recovery score")
ax.set_xlabel("set")
ax.set_ylabel("CGCS recovery")
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)

fig5_path = FIG_DIR / f"{NOTEBOOK_NUM}_recovery_scores.png"
fig.savefig(fig5_path, dpi=180, bbox_inches="tight")
plt.show()

fig5_path

## 12. Interpretation

1. **What remains under constraint?**  
   Sieve-retained values match the prime set.

2. **What drifts?**  
   Random controls drift away from prime structure even when they match final count or layer retention.

3. **What is recoverable?**  
   The sieve recovers prime identity. Random removal does not.

4. **What is the core result?**  
   Same count does not imply same structure.

5. **What should not be overclaimed?**  
   Random controls are useful comparisons, not models of prime generation.

In [ ]:
interpretation_lines = [
    f"# {NOTEBOOK_TITLE}",
    "",
    "## Constraint result",
    "",
    "This notebook compared deterministic sieve filtering against random removal controls.",
    "",
    "The key test is whether matched count or matched retention reproduces prime structure.",
    "",
    "## Sets compared",
    "",
    "1. sieve_primes: the reference prime set recovered by deterministic sieve filtering.",
    "2. random_count_matched: a random set with the same final count as the prime set.",
    "3. random_layer_matched: a random-control process that removes the same number of candidates at each sieve layer.",
    "",
    "## Recovery scores",
    "",
    f"- sieve recovery = {sieve_recovery:.6f}",
    f"- random-count-matched recovery = {random_count_recovery:.6f}",
    f"- random-layer-matched recovery = {random_layer_recovery:.6f}",
    "",
    "## Core result",
    "",
    "Same count does not imply same structure.",
    "",
    "Random removal can imitate density or retention count, but it does not recover prime residue structure, gap structure, or prime identity.",
    "",
    "## Remains under constraint",
    "",
    "The sieve-retained set remains under divisibility constraints and recovers primes exactly.",
    "",
    "## Drift",
    "",
    "Random controls drift away from prime structure despite matched final count or matched layer removals.",
    "",
    "## Recoverability",
    "",
    "Sieve filtering recovers prime identity exactly. Random controls do not.",
    "",
    "## Caution",
    "",
    "Random controls are comparison baselines. They do not represent a theory of prime generation.",
]

interpretation = "\n".join(interpretation_lines)

figure_paths = [fig1_path, fig2_path, fig3_path, fig4_path, fig5_path]
figure_titles = [
    "Residue comparison modulo 6",
    "Gap histogram comparison",
    "Density by scale",
    "Layer retention comparison",
    "Recovery score comparison",
]

figures_md = "\n\n## Figures\n\n"
for i, (fig, title) in enumerate(zip(figure_paths, figure_titles), start=1):
    figures_md += f"### Figure {i} — {title}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

print(interpretation + figures_md)

## 13. Export data, notes, math, and TeX

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
set_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_set_metrics.csv"
residue_path = DATA_DIR / f"{NOTEBOOK_NUM}_residue_comparison.csv"
gap_summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_gap_summary.csv"
density_path = DATA_DIR / f"{NOTEBOOK_NUM}_density_by_scale.csv"
layer_retention_path = DATA_DIR / f"{NOTEBOOK_NUM}_layer_retention.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
set_metrics_df.to_csv(set_metrics_path, index=False)
residue_comparison_df.to_csv(residue_path, index=False)
gap_summary_df.to_csv(gap_summary_path, index=False)
density_df.to_csv(density_path, index=False)
pd.concat(
    [
        sieve_layers_df.assign(process="sieve"),
        random_layer_df.assign(process="random_layer_matched")
    ],
    ignore_index=True
).to_csv(layer_retention_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "set_metrics": str(set_metrics_path),
        "residue_comparison": str(residue_path),
        "gap_summary": str(gap_summary_path),
        "density_by_scale": str(density_path),
        "layer_retention": str(layer_retention_path),
    },
    "docs": {
        "interpretation": str(interpretation_path),
        "design_notes": str(design_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_path.write_text(interpretation + figures_md + "\n", encoding="utf-8")

design_lines = [
    f"# Design Notes — {NOTEBOOK_TITLE}",
    "",
    "## Notebook role",
    "",
    "Notebook 05 follows Notebook 04 by testing whether random removal can imitate sieve-retained prime structure.",
    "",
    "Notebook 04 showed layered sieve mechanism.",
    "Notebook 05 uses random controls to show that matched retention does not imply matched structure.",
    "",
    "## Sets",
    "",
    "1. sieve_primes",
    "2. random_count_matched",
    "3. random_layer_matched",
    "",
    "## Measurements",
    "",
    "1. residue distribution modulo 6",
    "2. gap distribution",
    "3. density by scale",
    "4. layer retention",
    "5. recovery score against prime set",
    "",
    "## CGCS score",
    "",
    "CGCS_recovery(S) = |S ∩ P_N| / |P_N|.",
    "",
    "## Core claim",
    "",
    "Same count does not imply same structure.",
    "",
    "## Figures",
    "",
    "1. residue comparison modulo 6",
    "2. gap histogram comparison",
    "3. density by scale",
    "4. layer retention comparison",
    "5. recovery score comparison",
    "",
    "## Handoff",
    "",
    "Notebook 06 should test recoverability under partial observation.",
]

design_path.write_text("\n".join(design_lines) + "\n", encoding="utf-8")

summary_tex_lines = [
    rf"\section*{{{NOTEBOOK_TITLE}}}",
    "",
    r"This notebook compares deterministic sieve filtering against random removal controls.",
    "",
    r"The recovery score is",
    r"\[",
    r"CGCS_{\mathrm{recovery}}(S)=",
    r"\frac{|S\cap P_N|}{|P_N|}.",
    r"\]",
    "",
    rf"For $N={N_MAX:,}$:",
    r"\begin{itemize}",
    rf"  \item sieve recovery $= {sieve_recovery:.6f}$",
    rf"  \item random-count recovery $= {random_count_recovery:.6f}$",
    rf"  \item random-layer recovery $= {random_layer_recovery:.6f}$",
    r"\end{itemize}",
    "",
    r"Same count does not imply same structure.",
]

summary_tex_path.write_text("\n".join(summary_tex_lines) + "\n", encoding="utf-8")

math_tex_lines = [
    r"\documentclass{article}",
    r"\usepackage{amsmath}",
    r"\usepackage{amssymb}",
    r"\usepackage[margin=1in]{geometry}",
    "",
    r"\begin{document}",
    "",
    r"\section*{Math Notes: Random vs Sieve}",
    "",
    r"\subsection*{Prime set}",
    "",
    r"\[",
    r"P_N = \{p : p \le N\}.",
    r"\]",
    "",
    r"\subsection*{Recovery score}",
    "",
    r"For a candidate set $S$, define",
    r"\[",
    r"CGCS_{\mathrm{recovery}}(S) =",
    r"\frac{|S \cap P_N|}{|P_N|}.",
    r"\]",
    "",
    r"\subsection*{Recovery drift}",
    "",
    r"\[",
    r"drift_{\mathrm{recovery}}(S) = 1 - CGCS_{\mathrm{recovery}}(S).",
    r"\]",
    "",
    r"\subsection*{Density}",
    "",
    r"\[",
    r"density(S) = \frac{|S|}{N-1}.",
    r"\]",
    "",
    r"\subsection*{Core distinction}",
    "",
    r"\[",
    r"|S| = |P_N| \nRightarrow S = P_N.",
    r"\]",
    "",
    r"Same count does not imply same structure.",
    "",
    r"\end{document}",
]

math_tex_path.write_text("\n".join(math_tex_lines) + "\n", encoding="utf-8")

summary_path, set_metrics_path, residue_path, gap_summary_path, density_path, layer_retention_path, metadata_path, interpretation_path, design_path, summary_tex_path, math_tex_path

## 14. Export zip

Pi-stage-lab style root export zip, with optional Colab download lines left commented.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 15. Next notebook handoff

Next notebook:

```text
06_recoverability_under_partial_observation.ipynb
```

Purpose:

> corrupt or hide prime data and test which structures remain recoverable from partial observations.

In [ ]:
next_step = "Notebook 06: recoverability under partial observation."
print(next_step)